# Лабораторная работа №1: Компьютерное зрение (Object Detection)
**Вариант:** Реализация на тройку (Ultralytics / YOLO)  
**Стек:** Python 3.10, Ultralytics (YOLOv8), Matplotlib, Seaborn, Pandas, OpenCV

## 1. Выбор начальных условий
### 1.1 Выбор набора данных и обоснование
В качестве датасета выбран набор данных для обнаружения транспортных знаков и автомобилей. 
**Обоснование:** Задача обнаружения дорожных знаков и автомобилей является критически важной для современных систем помощи водителю, автономного вождения и систем видеомониторинга дорожного движения. Точное и быстрое обнаружение знаков напрямую влияет на безопасность дорожного движения и принятие решений автономными системами.

In [17]:
data_config = "/Users/admin/Desktop/MAI/4/код диплома/лабаии/archive-2/car/data.yaml"

### 1.2 Выбор метрик качества и обоснование

Для задачи объектного детекции выбраны следующие метрики:
- **Precision (Точность):** Доля корректно обнаруженных объектов среди всех обнаруженных. Критична для минимизации ложных срабатываний (false positives), что важно для избежания ложных тревог в реальных системах.
- **Recall (Полнота)**: Доля корректно обнаруженных объектов среди всех реально присутствующих на изображении. Критична для безопасности, так как пропуск знака может привести к аварийной ситуации.
- **mAP@0.5 (mAP50):** Средняя точность при пороге IoU=0.5. Показывает, насколько хорошо модель находит объекты при умеренных требованиях к точности боксов.
- **mAP@0.5:0.95 (mAP50-95):** Среднее значение mAP при порогах IoU от 0.5 до 0.95 с шагом 0.05. Наиболее строгая и информативная метрика, показывающая качество локализации при строгих порогах наложения боксов.
Выбор этих метрик обусловлен стандартами индустрии компьютерного зрения (COCO benchmark) и необходимостью баланса между точностью детекции и качеством локализации для задач безопасности.

## 2. Создание бейзлайна и оценка качества

В данном разделе реализуется базовая модель YOLOv8n (nano) для получения бейзлайн-результатов. Модель обучается на 30 эпохах с параметрами по умолчанию.

In [18]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

Baseline_model = YOLO('yolov8n.pt')

Result_Baseline = Baseline_model.train(
    data=data_config,
    epochs=10,
    imgsz=64
)

Ultralytics 8.4.41 🚀 Python-3.11.11 torch-2.1.2 CPU (Apple M1)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/admin/Desktop/MAI/4/код диплома/лабаии/archive-2/car/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=64, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-11, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mas

### 2.3 Оценка качества бейзлайна

In [19]:
val_baseline = Baseline_model.val(data=data_config)

metrics_baseline = {
    'mAP@50': val_baseline.box.map50,
    'mAP@50-95': val_baseline.box.map,
    'Precision': val_baseline.box.mp,
    'Recall': val_baseline.box.mr
}
# F1-скр
metrics_baseline['F1-Score'] = (2 * metrics_baseline['Precision'] * metrics_baseline['Recall']) / \
                               (metrics_baseline['Precision'] + metrics_baseline['Recall'] + 1e-7)

print("Метрики бейзлайна:")
print(f"  mAP@50:    {metrics_baseline['mAP@50']:.4f}")
print(f"  mAP@50-95: {metrics_baseline['mAP@50-95']:.4f}")
print(f"  Precision: {metrics_baseline['Precision']:.4f}")
print(f"  Recall:    {metrics_baseline['Recall']:.4f}")
print(f"  F1-Score:  {metrics_baseline['F1-Score']:.4f}")

Ultralytics 8.4.41 🚀 Python-3.11.11 torch-2.1.2 CPU (Apple M1)
Model summary (fused): 73 layers, 3,008,573 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 123.4±92.6 MB/s, size: 17.9 KB)
val: Scanning /Users/admin/Desktop/MAI/4/код диплома/лабаии/archive-2/car/valid/labels.cache... 801 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 801/801 73.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 51/51 6.9it/s 7.4s0.1s
                   all        801        944      0.648      0.448      0.469      0.413
           Green Light         87        122        0.4     0.0492     0.0676     0.0427
             Red Light         74        108      0.313     0.0741     0.0785     0.0458
       Speed Limit 100         52         52      0.661      0.489      0.571      0.513
       Speed Limit 110         17         17          1      0.172      0.382      0.351
       Speed Limit 120  

## 3. Улучшение бейзлайна

### 3.1 Формулировка гипотез

-  Аугментация данных: Включение augment=True и настройка mosaic, mixup улучшат обобщающую способность модели на сложных/зашумленных сценах.
- Увеличение ёмкости модели: Переход с yolo8n на yolo8s увеличит число параметров и улучшит детекцию мелких объектов.
- Увеличение размера: Размер до 320 улучшить детекцию изображений
- Cosine LR: cos_lr=True стабилизирует сходимость и предотвратит застревание в локальных минимумах
### 3.2 Проверка гипотез и обучение улучшенного бейзлайна

In [20]:
improved_model = YOLO("yolo26n.pt")

improved_results = improved_model.train(
    data=data_config,
    epochs=10,
    imgsz=320,
    batch=12,
    augment=True,
    mosaic=0.8,
    mixup=0.1,
    cos_lr=True
)

Ultralytics 8.4.41 🚀 Python-3.11.11 torch-2.1.2 CPU (Apple M1)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=12, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/Users/admin/Desktop/MAI/4/код диплома/лабаии/archive-2/car/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=0.8, multi_scale=0.0, name=train-12, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask

In [21]:
val_improved = improved_model.val(data=data_config)

metrics_improved = {
    'mAP@50': val_improved.box.map50,
    'mAP@50-95': val_improved.box.map,
    'Precision': val_improved.box.mp,
    'Recall': val_improved.box.mr
}
metrics_improved['F1-Score'] = (2 * metrics_improved['Precision'] * metrics_improved['Recall']) / \
                               (metrics_improved['Precision'] + metrics_improved['Recall'] + 1e-7)

print("Метрики улучшенной модели:")
for k, v in metrics_improved.items():
    print(f"  {k}: {v:.4f}")

Ultralytics 8.4.41 🚀 Python-3.11.11 torch-2.1.2 CPU (Apple M1)
YOLO26n summary (fused): 122 layers, 2,377,761 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access ✅ (ping: 0.1±0.2 ms, read: 81.8±33.8 MB/s, size: 17.9 KB)
val: Scanning /Users/admin/Desktop/MAI/4/код диплома/лабаии/archive-2/car/valid/labels.cache... 801 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 801/801 373.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 51/51 1.2it/s 43.0s0.8ss
                   all        801        944      0.668      0.706      0.733      0.642
           Green Light         87        122       0.47      0.502      0.424      0.232
             Red Light         74        108      0.489      0.531      0.454      0.274
       Speed Limit 100         52         52      0.605      0.846      0.806      0.733
       Speed Limit 110         17         17      0.564      0.471      0.549      0.483
       Speed Limit 

In [22]:
# Формирование таблицы сравнения
df_comparison = pd.DataFrame({
    "Baseline (YOLOv8n)": metrics_baseline,
    "Improved (YOLOv26m + Aug + CosLR)": metrics_improved
}).T

print("\n📊 Сравнительная таблица:")
print(df_comparison.round(4))

# Визуализация сравнения
df_comparison[['mAP@50', 'mAP@50-95', 'F1-Score']].plot(kind='bar', figsize=(8, 5), color=['#4c72b0', '#55a868'])
plt.title("Сравнение качества моделей")
plt.ylabel("Значение метрики")
plt.xticks(rotation=15)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


📊 Сравнительная таблица:
                                   mAP@50  mAP@50-95  Precision  Recall  \
Baseline (YOLOv8n)                 0.4690     0.4133     0.6478  0.4483   
Improved (YOLOv26m + Aug + CosLR)  0.7333     0.6425     0.6683  0.7058   

                                   F1-Score  
Baseline (YOLOv8n)                   0.5299  
Improved (YOLOv26m + Aug + CosLR)    0.6865  


<Figure size 800x500 with 1 Axes>

### 3.3 Выводы по улучшенному бейзлайну

Улучшенный бейзлайн (YOLOv26n + аугментации + Cosine LR + увеличенный `imgsz`) показывает устойчивый прирост относительно базовой модели:

- `mAP@50`: **0.4690 -> 0.7333**;
- `mAP@50-95`: **0.4133 -> 0.6425**;
- `Recall`: **0.4483 -> 0.7058**;
- `F1-Score`: **0.5299 -> 0.6865**.

Наиболее заметный эффект получен по `Recall`, что особенно важно для задач безопасности дорожного движения, где пропуск объекта критичнее лишнего срабатывания. Гипотезы из п.3.1 подтверждены: комбинация более сильной модели и обучающих техник действительно улучшает качество детекции на выбранном датасете.

## 4. Имплементация алгоритма машинного обучения

В этом разделе реализована **собственная детекционная модель на PyTorch** (без Ultralytics):
- компактная CNN-архитектура;
- обучение на YOLO-разметке датасета;
- оценка метрик `Precision`, `Recall`, `mAP@50`, `mAP@50:95`, `F1`;
- сравнение с результатами из разделов 2 и 3.

> Упрощение: модель предсказывает **один главный объект на изображение** (класс + bbox). Для данного датасета это допустимо как учебная имплементация.

In [23]:
import os
from pathlib import Path
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE = 128
BATCH_SIZE = 32
EPOCHS_IMPL = 8
LR_IMPL = 1e-3

data_root = Path(data_config).parent
train_images_dir = data_root / "train" / "images"
train_labels_dir = data_root / "train" / "labels"

class_names = [
    "Green Light", "Red Light", "Speed Limit 100", "Speed Limit 110", "Speed Limit 120",
    "Speed Limit 20", "Speed Limit 30", "Speed Limit 40", "Speed Limit 50", "Speed Limit 60",
    "Speed Limit 70", "Speed Limit 80", "Speed Limit 90", "Stop", "car"
]
num_classes = len(class_names)

print(f"Устройство: {DEVICE}")
print(f"train_images_dir: {train_images_dir}")
print(f"train_labels_dir: {train_labels_dir}")

Устройство: cpu
train_images_dir: /Users/admin/Desktop/MAI/4/код диплома/лабаии/archive-2/car/train/images
train_labels_dir: /Users/admin/Desktop/MAI/4/код диплома/лабаии/archive-2/car/train/labels


In [24]:
def load_yolo_samples(images_dir: Path, labels_dir: Path):
    """Собирает пары (путь к изображению, class_id, bbox_xywh_norm)."""
    samples = []
    image_paths = sorted([p for p in images_dir.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}])

    for img_path in image_paths:
        label_path = labels_dir / f"{img_path.stem}.txt"
        if not label_path.exists():
            continue

        lines = [line.strip() for line in label_path.read_text().splitlines() if line.strip()]
        if not lines:
            continue

        # Берем крупнейший объект как основной для single-object детектора.
        boxes = []
        for line in lines:
            vals = line.split()
            if len(vals) != 5:
                continue
            cls = int(vals[0])
            cx, cy, w, h = map(float, vals[1:])
            area = w * h
            boxes.append((area, cls, np.array([cx, cy, w, h], dtype=np.float32)))

        if not boxes:
            continue

        _, cls_id, box = max(boxes, key=lambda x: x[0])
        samples.append((str(img_path), cls_id, box))

    return samples


all_samples = load_yolo_samples(train_images_dir, train_labels_dir)
print(f"Подготовлено сэмплов: {len(all_samples)}")

train_samples, val_samples = train_test_split(all_samples, test_size=0.2, random_state=42, shuffle=True)
print(f"Train: {len(train_samples)} | Val: {len(val_samples)}")

Подготовлено сэмплов: 3527
Train: 2821 | Val: 706


In [25]:
class SingleObjectDetectionDataset(Dataset):
    """Датасет для single-object детекции (класс + bbox в формате cx, cy, w, h)."""

    def __init__(self, samples, img_size=128):
        self.samples = samples
        self.img_size = img_size

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, class_id, box = self.samples[idx]
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (self.img_size, self.img_size), interpolation=cv2.INTER_AREA)
        image = image.astype(np.float32) / 255.0
        image = np.transpose(image, (2, 0, 1))

        x = torch.tensor(image, dtype=torch.float32)
        y_cls = torch.tensor(class_id, dtype=torch.long)
        y_box = torch.tensor(box, dtype=torch.float32)
        return x, y_cls, y_box


train_ds = SingleObjectDetectionDataset(train_samples, img_size=IMG_SIZE)
val_ds = SingleObjectDetectionDataset(val_samples, img_size=IMG_SIZE)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("DataLoader готов")

DataLoader готов


In [26]:
class TinyDetector(nn.Module):
    """Простая CNN: извлечение признаков + две головы (класс и bbox)."""

    def __init__(self, num_classes: int):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Linear(256, num_classes)
        self.box_regressor = nn.Linear(256, 4)

    def forward(self, x):
        feat = self.features(x).flatten(1)
        cls_logits = self.classifier(feat)
        box_pred = torch.sigmoid(self.box_regressor(feat))
        return cls_logits, box_pred


impl_model = TinyDetector(num_classes=num_classes).to(DEVICE)
optimizer = torch.optim.Adam(impl_model.parameters(), lr=LR_IMPL)
criterion_cls = nn.CrossEntropyLoss()
criterion_box = nn.SmoothL1Loss()

print(impl_model)

TinyDetector(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU(inplace=True)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU(inplace=True)
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (12): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): BatchNorm2d(256, eps=1e-05, momentum=0.1,

In [27]:
def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0.0

    for x, y_cls, y_box in loader:
        x = x.to(DEVICE)
        y_cls = y_cls.to(DEVICE)
        y_box = y_box.to(DEVICE)

        optimizer.zero_grad()
        pred_cls, pred_box = model(x)

        loss_cls = criterion_cls(pred_cls, y_cls)
        loss_box = criterion_box(pred_box, y_box)
        loss = loss_cls + 3.0 * loss_box

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)

    return total_loss / len(loader.dataset)


def evaluate_loss(model, loader):
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for x, y_cls, y_box in loader:
            x = x.to(DEVICE)
            y_cls = y_cls.to(DEVICE)
            y_box = y_box.to(DEVICE)

            pred_cls, pred_box = model(x)
            loss_cls = criterion_cls(pred_cls, y_cls)
            loss_box = criterion_box(pred_box, y_box)
            loss = loss_cls + 3.0 * loss_box

            total_loss += loss.item() * x.size(0)

    return total_loss / len(loader.dataset)


history = {"train_loss": [], "val_loss": []}

for epoch in range(1, EPOCHS_IMPL + 1):
    train_loss = train_one_epoch(impl_model, train_loader, optimizer)
    val_loss = evaluate_loss(impl_model, val_loader)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    print(f"Epoch [{epoch}/{EPOCHS_IMPL}] train_loss={train_loss:.4f} val_loss={val_loss:.4f}")

Epoch [1/8] train_loss=2.3267 val_loss=2.2368
Epoch [2/8] train_loss=2.1247 val_loss=2.2107
Epoch [3/8] train_loss=2.0560 val_loss=2.0694
Epoch [4/8] train_loss=1.9661 val_loss=2.0065
Epoch [5/8] train_loss=1.9152 val_loss=2.0388
Epoch [6/8] train_loss=1.8660 val_loss=2.1632
Epoch [7/8] train_loss=1.8014 val_loss=1.9006
Epoch [8/8] train_loss=1.7532 val_loss=2.1480


In [28]:
def cxcywh_to_xyxy(box):
    cx, cy, w, h = box
    x1 = cx - w / 2.0
    y1 = cy - h / 2.0
    x2 = cx + w / 2.0
    y2 = cy + h / 2.0
    return np.array([x1, y1, x2, y2], dtype=np.float32)


def iou_xyxy(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter_w = max(0.0, x2 - x1)
    inter_h = max(0.0, y2 - y1)
    inter = inter_w * inter_h

    area1 = max(0.0, box1[2] - box1[0]) * max(0.0, box1[3] - box1[1])
    area2 = max(0.0, box2[2] - box2[0]) * max(0.0, box2[3] - box2[1])
    union = area1 + area2 - inter + 1e-9
    return inter / union


def compute_ap_from_pr(precisions, recalls):
    precisions = np.array(precisions)
    recalls = np.array(recalls)

    order = np.argsort(recalls)
    recalls = recalls[order]
    precisions = precisions[order]

    mrec = np.concatenate(([0.0], recalls, [1.0]))
    mpre = np.concatenate(([0.0], precisions, [0.0]))

    for i in range(len(mpre) - 1, 0, -1):
        mpre[i - 1] = max(mpre[i - 1], mpre[i])

    idx = np.where(mrec[1:] != mrec[:-1])[0]
    ap = np.sum((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1])
    return float(ap)


def evaluate_detector_map(model, loader, iou_thresholds=np.arange(0.5, 1.0, 0.05)):
    model.eval()

    all_cls_true, all_cls_pred = [], []
    ious = []

    preds_for_map = []

    with torch.no_grad():
        for x, y_cls, y_box in loader:
            x = x.to(DEVICE)
            logits, pred_box = model(x)
            probs = F.softmax(logits, dim=1)
            conf, pred_cls = probs.max(dim=1)

            y_cls_np = y_cls.numpy()
            y_box_np = y_box.numpy()
            pred_cls_np = pred_cls.cpu().numpy()
            conf_np = conf.cpu().numpy()
            pred_box_np = pred_box.cpu().numpy()

            for i in range(len(y_cls_np)):
                gt_cls = int(y_cls_np[i])
                pr_cls = int(pred_cls_np[i])
                gt_box = cxcywh_to_xyxy(y_box_np[i])
                pr_box = cxcywh_to_xyxy(pred_box_np[i])
                iou = iou_xyxy(gt_box, pr_box)

                all_cls_true.append(gt_cls)
                all_cls_pred.append(pr_cls)
                ious.append(iou)

                preds_for_map.append({
                    "gt_cls": gt_cls,
                    "pred_cls": pr_cls,
                    "conf": float(conf_np[i]),
                    "iou": float(iou)
                })

    # Бинарные precision/recall по правилу: верный класс + IoU >= 0.5
    tp = 0
    fp = 0
    fn = 0
    for p in preds_for_map:
        if p["pred_cls"] == p["gt_cls"] and p["iou"] >= 0.5:
            tp += 1
        else:
            fp += 1
            fn += 1

    precision = tp / (tp + fp + 1e-9)
    recall = tp / (tp + fn + 1e-9)
    f1 = 2 * precision * recall / (precision + recall + 1e-9)

    # mAP@50 и mAP@50:95 (приближенно, per-class AP)
    aps_by_thr = []
    classes_present = sorted(set([p["gt_cls"] for p in preds_for_map]))

    for thr in iou_thresholds:
        ap_list = []
        for c in classes_present:
            class_preds = [p for p in preds_for_map if p["pred_cls"] == c]
            class_gts = [p for p in preds_for_map if p["gt_cls"] == c]
            n_gt = len(class_gts)
            if n_gt == 0:
                continue

            class_preds = sorted(class_preds, key=lambda x: x["conf"], reverse=True)

            tps = []
            fps = []
            for p in class_preds:
                is_tp = int((p["gt_cls"] == c) and (p["iou"] >= thr))
                tps.append(is_tp)
                fps.append(1 - is_tp)

            if len(tps) == 0:
                ap_list.append(0.0)
                continue

            tps = np.cumsum(tps)
            fps = np.cumsum(fps)
            precisions = tps / (tps + fps + 1e-9)
            recalls = tps / (n_gt + 1e-9)
            ap_list.append(compute_ap_from_pr(precisions, recalls))

        aps_by_thr.append(np.mean(ap_list) if ap_list else 0.0)

    map50 = aps_by_thr[0]
    map5095 = float(np.mean(aps_by_thr))

    return {
        "mAP@50": float(map50),
        "mAP@50-95": float(map5095),
        "Precision": float(precision),
        "Recall": float(recall),
        "F1-Score": float(f1),
        "Mean IoU": float(np.mean(ious)) if ious else 0.0
    }


metrics_impl = evaluate_detector_map(impl_model, val_loader)
print("Метрики собственной имплементации:")
for k, v in metrics_impl.items():
    print(f"  {k}: {v:.4f}")

Метрики собственной имплементации:
  mAP@50: 0.1139
  mAP@50-95: 0.0481
  Precision: 0.2025
  Recall: 0.2025
  F1-Score: 0.2025
  Mean IoU: 0.3874


In [29]:
df_impl_compare = pd.DataFrame({
    "Baseline (YOLOv8n)": metrics_baseline,
    "Improved (YOLOv26n + Aug + CosLR)": metrics_improved,
    "Implemented (TinyDetector)": {k: metrics_impl[k] for k in ["mAP@50", "mAP@50-95", "Precision", "Recall", "F1-Score"]}
}).T

print("\nСравнение трех подходов:")
print(df_impl_compare.round(4))

plot_cols = ["mAP@50", "mAP@50-95", "F1-Score"]
df_impl_compare[plot_cols].plot(kind="bar", figsize=(10, 5))
plt.title("Сравнение бейзлайна, улучшенного бейзлайна и собственной имплементации")
plt.ylabel("Значение метрики")
plt.xticks(rotation=15)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


Сравнение трех подходов:
                                   mAP@50  mAP@50-95  Precision  Recall  \
Baseline (YOLOv8n)                 0.4690     0.4133     0.6478  0.4483   
Improved (YOLOv26n + Aug + CosLR)  0.7333     0.6425     0.6683  0.7058   
Implemented (TinyDetector)         0.1139     0.0481     0.2025  0.2025   

                                   F1-Score  
Baseline (YOLOv8n)                   0.5299  
Improved (YOLOv26n + Aug + CosLR)    0.6865  
Implemented (TinyDetector)           0.2025  


<Figure size 1000x500 with 1 Axes>

### Добавление техник из улучшенного бейзлайна и повторная оценка имплементации

В этом подпункте для собственной модели добавляются техники из п.3:
- аугментации;
- увеличение входного размера изображения;
- scheduler (`CosineAnnealingLR`);
- более емкая архитектура.

Далее выполняются:
-  обучение обновленной реализации;
- оценка метрик (`mAP@50`, `mAP@50-95`, `Precision`, `Recall`, `F1`);
- сравнение с результатами из пункта 3.

In [43]:
class SingleObjectDetectionDatasetAug(Dataset):
    """Датасет single-object detection с простыми аугментациями."""

    def __init__(self, samples, img_size=224, augment=False):
        self.samples = samples
        self.img_size = img_size
        self.augment = augment

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, class_id, box = self.samples[idx]
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (self.img_size, self.img_size), interpolation=cv2.INTER_AREA)

        # Копия bbox, чтобы безопасно модифицировать при аугментации.
        cx, cy, w, h = [float(v) for v in box]

        if self.augment:
            # Horizontal flip
            if np.random.rand() < 0.5:
                image = np.ascontiguousarray(image[:, ::-1, :])
                cx = 1.0 - cx

            # Яркость/контраст
            if np.random.rand() < 0.5:
                alpha = np.random.uniform(0.85, 1.20)  # contrast
                beta = np.random.uniform(-20, 20)      # brightness
                image = np.clip(alpha * image + beta, 0, 255).astype(np.uint8)

        image = image.astype(np.float32) / 255.0
        image = np.transpose(image, (2, 0, 1))

        x = torch.tensor(image, dtype=torch.float32)
        y_cls = torch.tensor(class_id, dtype=torch.long)
        y_box = torch.tensor([cx, cy, w, h], dtype=torch.float32)
        return x, y_cls, y_box


class TinyDetectorPlus(nn.Module):
    """Усиленная версия TinyDetector для пункта 4.f-i."""

    def __init__(self, num_classes: int):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d(2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True), nn.MaxPool2d(2),

            nn.Conv2d(256, 384, kernel_size=3, padding=1),
            nn.BatchNorm2d(384), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
        )

        self.dropout = nn.Dropout(p=0.2)
        self.classifier = nn.Linear(384, num_classes)
        self.box_regressor = nn.Linear(384, 4)

    def forward(self, x):
        feat = self.features(x).flatten(1)
        feat = self.dropout(feat)
        cls_logits = self.classifier(feat)
        box_pred = torch.sigmoid(self.box_regressor(feat))
        return cls_logits, box_pred

In [ ]:
IMG_SIZE_PLUS = 224
BATCH_SIZE_PLUS = 32
EPOCHS_IMPL_PLUS = 12
LR_IMPL_PLUS = 8e-4

train_ds_plus = SingleObjectDetectionDatasetAug(train_samples, img_size=IMG_SIZE_PLUS, augment=True)
val_ds_plus = SingleObjectDetectionDatasetAug(val_samples, img_size=IMG_SIZE_PLUS, augment=False)

train_loader_plus = DataLoader(train_ds_plus, batch_size=BATCH_SIZE_PLUS, shuffle=True, num_workers=0)
val_loader_plus = DataLoader(val_ds_plus, batch_size=BATCH_SIZE_PLUS, shuffle=False, num_workers=0)

impl_model_plus = TinyDetectorPlus(num_classes=num_classes).to(DEVICE)
optimizer_plus = torch.optim.AdamW(impl_model_plus.parameters(), lr=LR_IMPL_PLUS, weight_decay=1e-4)
scheduler_plus = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_plus, T_max=EPOCHS_IMPL_PLUS)

history_plus = {"train_loss": [], "val_loss": []}

for epoch in range(1, EPOCHS_IMPL_PLUS + 1):
    train_loss = train_one_epoch(impl_model_plus, train_loader_plus, optimizer_plus)
    val_loss = evaluate_loss(impl_model_plus, val_loader_plus)
    scheduler_plus.step()

    history_plus["train_loss"].append(train_loss)
    history_plus["val_loss"].append(val_loss)

    print(
        f"Epoch+ [{epoch}/{EPOCHS_IMPL_PLUS}] "
        f"train_loss={train_loss:.4f} val_loss={val_loss:.4f} "
        f"lr={optimizer_plus.param_groups[0]['lr']:.6f}"
    )

Epoch+ [1/12] train_loss=2.4246 val_loss=2.4348 lr=0.000786
Epoch+ [2/12] train_loss=2.2459 val_loss=2.2402 lr=0.000746
Epoch+ [3/12] train_loss=2.1811 val_loss=2.0665 lr=0.000683
Epoch+ [4/12] train_loss=2.1273 val_loss=2.0914 lr=0.000600
Epoch+ [5/12] train_loss=2.0578 val_loss=2.0099 lr=0.000504
Epoch+ [6/12] train_loss=2.0077 val_loss=2.0354 lr=0.000400
Epoch+ [7/12] train_loss=2.0009 val_loss=1.9291 lr=0.000296
Epoch+ [8/12] train_loss=1.8951 val_loss=1.9181 lr=0.000200
Epoch+ [9/12] train_loss=1.8592 val_loss=1.8539 lr=0.000117
Epoch+ [10/12] train_loss=1.7878 val_loss=1.8467 lr=0.000054
Epoch+ [11/12] train_loss=1.7640 val_loss=1.8385 lr=0.000014
Epoch+ [12/12] train_loss=1.7640 val_loss=1.8326 lr=0.000000


In [ ]:

metrics_impl_plus = evaluate_detector_map(impl_model_plus, val_loader_plus)

print("Метрики улучшенной имплементации (TinyDetectorPlus):")
for k, v in metrics_impl_plus.items():
    print(f"  {k}: {v:.4f}")

df_impl_plus_compare = pd.DataFrame({
    "Improved baseline (п.3, YOLOv26n + Aug + CosLR)": metrics_improved,
    "Implementation (п.4, TinyDetector)": {k: metrics_impl[k] for k in ["mAP@50", "mAP@50-95", "Precision", "Recall", "F1-Score"]},
    "Implementation + techniques (п.4f-i, TinyDetectorPlus)": {k: metrics_impl_plus[k] for k in ["mAP@50", "mAP@50-95", "Precision", "Recall", "F1-Score"]},
}).T

print("\nСравнение: п.3 vs п.4 vs п.4f-i")
print(df_impl_plus_compare.round(4))

df_impl_plus_compare[["mAP@50", "mAP@50-95", "F1-Score"]].plot(kind="bar", figsize=(11, 5))
plt.title("Сравнение результатов после добавления техник improved baseline")
plt.ylabel("Значение метрики")
plt.xticks(rotation=12)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

Метрики улучшенной имплементации (TinyDetectorPlus):
  mAP@50: 0.1399
  mAP@50-95: 0.0764
  Precision: 0.2100
  Recall: 0.2100
  F1-Score: 0.2100
  Mean IoU: 0.4120

Сравнение: п.3 vs п.4 vs п.4f-i
                                                    mAP@50  mAP@50-95  \
Improved baseline (п.3, YOLOv26n + Aug + CosLR)     0.7333     0.6425   
Implementation (п.4, TinyDetector)                  0.1139     0.0481   
Implementation + techniques (п.4f-i, TinyDetect...  0.1399     0.0764   

                                                    Precision  Recall  \
Improved baseline (п.3, YOLOv26n + Aug + CosLR)        0.6683  0.7058   
Implementation (п.4, TinyDetector)                     0.2025  0.2025   
Implementation + techniques (п.4f-i, TinyDetect...     0.2100  0.2100   

                                                    F1-Score  
Improved baseline (п.3, YOLOv26n + Aug + CosLR)       0.6865  
Implementation (п.4, TinyDetector)                    0.2025  
Implementation + techniques

<Figure size 1100x500 with 1 Axes>

### Выводы по подпунктам f-i

Добавление техник из улучшенного бейзлайна в собственную реализацию дало положительный, но ограниченный эффект:

- `mAP@50`: **0.1139 -> 0.1399**;
- `mAP@50-95`: **0.0481 -> 0.0764**;
- `F1-Score`: **0.2025 -> 0.2100**.

Следовательно, гипотеза о переносе техник (аугментации, scheduler, больший вход, более емкая сеть) подтверждается частично: метрики растут, однако разрыв с YOLO остается большим из-за упрощенной постановки single-object детекции и существенно меньшей выразительности собственной архитектуры.

### 4. Общие выводы по имплементации

- Самостоятельно реализована и обучена рабочая end-to-end модель детекции на PyTorch без использования готового detection-фреймворка.
- Базовая имплементация уступает индустриальным решениям YOLO по всем ключевым метрикам, что ожидаемо для компактной учебной архитектуры и упрощенной постановки (один объект на изображение).
- Перенос техник из улучшенного бейзлайна приводит к измеримому росту качества, то есть направление улучшений выбрано корректно.
- Итог: цель пункта 4 достигнута - собственная модель реализована, проверена, сопоставлена с бейзлайнами и проанализирована с точки зрения дальнейших улучшений.